# Comprehensive SiteX Data Engineering Pipeline

This notebook implements a fully normalized relational database schema for SiteX data.
It **automatically discovers and processes all `.json` files** in the current folder,
merges them, deduplicates by `place_id`, and writes to **`ktm_all.db`**.

Tables produced:
- `place`: Core information (deduplicated)
- `categories_detailed`: All categories associated with a place
- `images`: Summary images
- `images_detailed`: Full list of images with titles
- `open_hours_detailed`: Daily schedule and intervals
- `owner_info`: Details about the business owner
- `popular_times_detailed`: Hourly popularity data by day
- `ratings_distribution`: 1-5 star review counts
- `reviews`: Primary review data
- `reviews_detailed`: Extended review metadata

In [ ]:
import sqlite3
import json
import os
import glob
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import ast
import re

# --- Helper Functions ---

def parse_time_string(time_str):
    if not isinstance(time_str, str):
        return None
    time_str = time_str.replace(' AM', ' AM').replace(' PM', ' PM').strip().replace('.', '')
    formats = ["%I:%M %p", "%I %p"]
    for fmt in formats:
        try:
            dt_object = datetime.strptime(time_str, fmt)
            return timedelta(hours=dt_object.hour, minutes=dt_object.minute)
        except ValueError:
            pass
    return None

def calculate_duration_from_intervals(intervals):
    total_duration_minutes = 0
    if not intervals or not isinstance(intervals, list):
        return 0.0
    for interval_str in intervals:
        if not isinstance(interval_str, str):
            continue
        interval_str = interval_str.replace('\u2013', '-').replace('\u2014', '-')
        if 'Open 24 hours' in interval_str or '24 hours' in interval_str.lower():
            total_duration_minutes += 24 * 60
            continue
        if 'Closed' in interval_str:
            continue
        parts = interval_str.split('-')
        if len(parts) != 2:
            continue
        try:
            start_td = parse_time_string(parts[0].strip())
            end_td = parse_time_string(parts[1].strip())
            if start_td is None or end_td is None:
                continue
            duration = end_td - start_td
            if duration < timedelta(0):
                duration += timedelta(days=1)
            total_duration_minutes += duration.total_seconds() / 60
        except ValueError:
            continue
    return total_duration_minutes / 60.0

def calculate_total_open_hours(open_hours_data):
    if not open_hours_data or not isinstance(open_hours_data, dict):
        return float('nan')
    daily_hours = {}
    for day, intervals in open_hours_data.items():
        daily_hours[day] = calculate_duration_from_intervals(intervals)
    total_sum_hours = sum(daily_hours.values())
    num_non_zero_days = sum(1 for h in daily_hours.values() if h > 0)
    if num_non_zero_days == 1:
        return total_sum_hours * 6
    return total_sum_hours

def clean_value(val):
    if isinstance(val, str):
        return val.replace('`', '').strip()
    return val

## Data Loading — All JSON Files in This Folder

In [ ]:
# ── Discover all .json files in the same directory as this notebook ──────────
notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
json_files = sorted(glob.glob(os.path.join(notebook_dir, '*.json')))

if not json_files:
    raise FileNotFoundError(f"No .json files found in: {notebook_dir}")

print(f"Found {len(json_files)} JSON file(s) in: {notebook_dir}")
for jf in json_files:
    size_mb = os.path.getsize(jf) / (1024 * 1024)
    print(f"  - {os.path.basename(jf)}  ({size_mb:.1f} MB)")

# ── Load and concatenate all files ───────────────────────────────────────────
dfs = []
for jf in json_files:
    print(f"\nLoading: {os.path.basename(jf)} ...")
    with open(jf, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df_part = pd.DataFrame(data)
    df_part['_source_file'] = os.path.basename(jf)   # track origin
    print(f"  Records loaded: {len(df_part):,}")
    dfs.append(df_part)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal records (all files combined): {len(df_raw):,}")

# ── Deduplicate by place_id (prefer first occurrence) ────────────────────────
before = len(df_raw)
if 'place_id' in df_raw.columns:
    df_raw = df_raw.drop_duplicates(subset=['place_id'], keep='first').reset_index(drop=True)
elif 'input_id' in df_raw.columns:
    df_raw = df_raw.drop_duplicates(subset=['input_id'], keep='first').reset_index(drop=True)

print(f"Duplicates removed: {before - len(df_raw):,}")
print(f"Records after deduplication: {len(df_raw):,}")

## Normalization and Relational Database Export → `ktm_all.db`

In [ ]:
# Output to the shared merged database
db_file = os.path.join(notebook_dir, '..', 'DataEngineering', 'ktm_all.db')
db_file = os.path.normpath(db_file)
print(f"Target database: {db_file}")

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# 1. Define Tables Schema
cursor.executescript("""
    DROP TABLE IF EXISTS place;
    DROP TABLE IF EXISTS categories_detailed;
    DROP TABLE IF EXISTS images;
    DROP TABLE IF EXISTS images_detailed;
    DROP TABLE IF EXISTS open_hours_detailed;
    DROP TABLE IF EXISTS owner_info;
    DROP TABLE IF EXISTS popular_times_detailed;
    DROP TABLE IF EXISTS ratings_distribution;
    DROP TABLE IF EXISTS reviews;
    DROP TABLE IF EXISTS reviews_detailed;

    CREATE TABLE place (
        place_id TEXT PRIMARY KEY,
        input_id TEXT,
        title TEXT,
        category TEXT,
        address TEXT,
        latitude REAL,
        longitude REAL,
        phone TEXT,
        plus_code TEXT,
        review_count INTEGER,
        review_rating REAL,
        price_range TEXT,
        total_open_hours REAL,
        city TEXT,
        borough TEXT,
        street TEXT,
        postal_code TEXT,
        timezone TEXT,
        status TEXT,
        description TEXT,
        link TEXT,
        thumbnail TEXT,
        menu_link TEXT,
        source_file TEXT
    );

    CREATE TABLE categories_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        category_name TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE images (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        image_url TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE images_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        title TEXT,
        image_url TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE open_hours_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        day TEXT,
        hours TEXT,
        duration_hours REAL,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE owner_info (
        place_id TEXT PRIMARY KEY,
        owner_id TEXT,
        owner_name TEXT,
        owner_link TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE popular_times_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        day TEXT,
        hour INTEGER,
        popularity INTEGER,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE ratings_distribution (
        place_id TEXT PRIMARY KEY,
        rating_1 INTEGER,
        rating_2 INTEGER,
        rating_3 INTEGER,
        rating_4 INTEGER,
        rating_5 INTEGER,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE reviews (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        reviewer_name TEXT,
        rating INTEGER,
        description TEXT,
        date TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );

    CREATE TABLE reviews_detailed (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        place_id TEXT,
        reviewer_name TEXT,
        rating INTEGER,
        description TEXT,
        date TEXT,
        profile_picture TEXT,
        images TEXT,
        FOREIGN KEY(place_id) REFERENCES place(place_id)
    );
""")

# 2. Populate Tables
total = len(df_raw)
for i, (_, row) in enumerate(df_raw.iterrows()):
    if i % 1000 == 0:
        print(f"  Processing row {i:,} / {total:,} ...", end='\r')

    pid = row.get('place_id') or row.get('input_id')

    # --- Place Table ---
    addr = row.get('complete_address', {})
    if not isinstance(addr, dict):
        addr = {}
    place_data = (
        pid, row.get('input_id'), row.get('title'), row.get('category'),
        row.get('address'), row.get('latitude'), row.get('longtitude'),
        row.get('phone'), row.get('plus_code'), row.get('review_count'),
        row.get('review_rating'), row.get('price_range'),
        calculate_total_open_hours(row.get('open_hours')),
        addr.get('city'), addr.get('borough'), addr.get('street'), addr.get('postal_code'),
        row.get('timezone'), row.get('status'), row.get('description'),
        clean_value(row.get('link')), clean_value(row.get('thumbnail')),
        clean_value(row.get('menu', {}).get('link')) if isinstance(row.get('menu'), dict) else None,
        row.get('_source_file')
    )
    cursor.execute("INSERT OR IGNORE INTO place VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", place_data)

    # --- Categories Detailed ---
    cats = row.get('categories', [])
    if isinstance(cats, list):
        for c in cats:
            cursor.execute("INSERT INTO categories_detailed (place_id, category_name) VALUES (?, ?)", (pid, c))

    # --- Images ---
    thumb = clean_value(row.get('thumbnail'))
    if thumb:
        cursor.execute("INSERT INTO images (place_id, image_url) VALUES (?, ?)", (pid, thumb))

    # --- Images Detailed ---
    imgs = row.get('images', [])
    if isinstance(imgs, list):
        for img in imgs:
            if isinstance(img, dict):
                cursor.execute("INSERT INTO images_detailed (place_id, title, image_url) VALUES (?, ?, ?)",
                               (pid, img.get('title'), clean_value(img.get('image'))))

    # --- Open Hours Detailed ---
    oh = row.get('open_hours', {})
    if isinstance(oh, dict):
        for day, intervals in oh.items():
            duration = calculate_duration_from_intervals(intervals)
            cursor.execute("INSERT INTO open_hours_detailed (place_id, day, hours, duration_hours) VALUES (?, ?, ?, ?)",
                           (pid, day, ", ".join(intervals) if isinstance(intervals, list) else str(intervals), duration))

    # --- Owner Info ---
    owner = row.get('owner', {})
    if isinstance(owner, dict) and owner:
        cursor.execute("INSERT OR IGNORE INTO owner_info VALUES (?, ?, ?, ?)",
                       (pid, owner.get('id'), owner.get('name'), clean_value(owner.get('link'))))

    # --- Popular Times Detailed ---
    pt = row.get('popular_times', {})
    if isinstance(pt, dict):
        for day, hours in pt.items():
            if isinstance(hours, dict):
                for h, pop in hours.items():
                    cursor.execute("INSERT INTO popular_times_detailed (place_id, day, hour, popularity) VALUES (?, ?, ?, ?)",
                                   (pid, day, int(h), int(pop)))

    # --- Ratings Distribution ---
    rd = row.get('reviews_per_rating', {})
    if isinstance(rd, dict) and rd:
        cursor.execute("INSERT OR IGNORE INTO ratings_distribution VALUES (?, ?, ?, ?, ?, ?)",
                       (pid, rd.get('1', 0), rd.get('2', 0), rd.get('3', 0), rd.get('4', 0), rd.get('5', 0)))

    # --- Reviews ---
    revs = row.get('user_reviews', [])
    if isinstance(revs, list):
        for r in revs:
            if isinstance(r, dict):
                cursor.execute("INSERT INTO reviews (place_id, reviewer_name, rating, description, date) VALUES (?, ?, ?, ?, ?)",
                               (pid, r.get('Name'), r.get('Rating'), r.get('Description'), r.get('When')))

    # --- Reviews Detailed ---
    revs_ext = row.get('user_reviews_extended', [])
    combined_revs = (revs if isinstance(revs, list) else []) + (revs_ext if isinstance(revs_ext, list) else [])
    for r in combined_revs:
        if isinstance(r, dict):
            cursor.execute("INSERT INTO reviews_detailed (place_id, reviewer_name, rating, description, date, profile_picture, images) VALUES (?, ?, ?, ?, ?, ?, ?)",
                           (pid, r.get('Name'), r.get('Rating'), r.get('Description'), r.get('When'),
                            clean_value(r.get('ProfilePicture')), str(r.get('Images', []))))

    # Commit every 500 rows to avoid holding too much in memory
    if i % 500 == 0:
        conn.commit()

conn.commit()
conn.close()
print(f"\n\nDatabase finalized: {db_file}")

## Verification
List all tables, record counts, and source-file breakdown.

In [ ]:
conn = sqlite3.connect(db_file)

tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", conn)
print("Tables in Database:")
display(tables)

print("\nRecord counts:")
for t in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {t}", conn)['count'][0]
    print(f"  - {t}: {count:,} records")

# Show breakdown by source JSON file
print("\nPlace records by source file:")
try:
    src_df = pd.read_sql_query(
        "SELECT source_file, COUNT(*) as places FROM place GROUP BY source_file ORDER BY places DESC",
        conn
    )
    display(src_df)
except Exception as e:
    print(f"  (could not read source_file column: {e})")

conn.close()

## Category Analysis: Finding Food & Dining Categories

This section queries the consolidated database (`ktm_all.db`), retrieves all unique category entries, and filters them using keyword matching to identify categories related to **cafes, restaurants, bars, and dining establishments**.

In [ ]:
import sqlite3
import pandas as pd
import re
import os

# Connect to consolidated database
notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
db_file = os.path.join(notebook_dir, '..', 'DataEngineering', 'ktm_all.db')
db_file = os.path.normpath(db_file)

print(f"Reading categories from: {db_file}")
conn = sqlite3.connect(db_file)

# Query unique categories and their occurrences
query = """
SELECT category_name, COUNT(*) as count 
FROM categories_detailed 
WHERE category_name IS NOT NULL
GROUP BY category_name 
ORDER BY count DESC
"""
df_categories = pd.read_sql_query(query, conn)
conn.close()

print(f"███ Total unique categories found in database: {len(df_categories):,}")
print(f"Top 20 most frequent categories in ktm_all.db:\n")
print(df_categories.head(20).to_string(index=False))

# Heuristic keyword patterns to match food, beverage, and dining types
cafe_keywords = r'\\bcafe\\b|coffee|tea|espresso|bakery|baker|pastry|dessert|ice cream|donut|cake|sweet|chocolate'
restaurant_keywords = r'restaurant|diner|bistr|grill|steak|pizza|burger|sushi|food|\\bmomo\\b|noodle|asian|indian|nepalese|italian|mexican|chinese|seafood|buffet|eatery|dhaba|canteen|kitchen'
bar_keywords = r'\\bbar\\b|\\bpub\\b|lounge|brewery|tavern|nightclub|wine|beer'

# Exclude list to filter out non-dining establishments, suppliers, education, and services
exclude_keywords = r'supplier|supply|repair|equipment|association|school|college|academy|class|consultant|wholesale|retail|importer|exporter|museum|library|institution|organization|government|office|bank|hospital|clinic|pharmacy|salon|studio|gym'

def analyze_category(cat):
    cat_lower = str(cat).lower()
    
    if re.search(exclude_keywords, cat_lower):
        return None
        
    matched_groups = []
    if re.search(cafe_keywords, cat_lower):
        matched_groups.append("Cafe/Bakery/Beverage")
    if re.search(restaurant_keywords, cat_lower):
        matched_groups.append("Restaurant/Dining")
    if re.search(bar_keywords, cat_lower):
        matched_groups.append("Bar/Pub/Lounge")
        
    if matched_groups:
        return " & ".join(matched_groups)
    return None

df_categories['matched_type'] = df_categories['category_name'].apply(analyze_category)
df_matched = df_categories[df_categories['matched_type'].notna()].copy()

print(f"\n███ Identified {len(df_matched)} dining-related categories.")

# Group breakdown
breakdown = df_matched.groupby('matched_type')['count'].agg(['count', 'sum']).rename(
    columns={'count': 'unique_categories', 'sum': 'total_establishments'}
)
print("\nDining Category Group Breakdown:\n")
print(breakdown.to_string())

print("\nList of identified categories (sorted by frequency):\n")
pd.set_option('display.max_rows', 150)
print(df_matched[['category_name', 'matched_type', 'count']].to_string(index=False))


## Clean Up: Drop Zero-Review Places in Dining Categories

For the curated list of food/cafe/restaurant categories, any place with **zero reviews** is not useful for analysis.
This cell finds those places and deletes them (and all their related rows in child tables) from `ktm_all.db`.

In [ ]:
import sqlite3
import os

notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
db_file = os.path.normpath(os.path.join(notebook_dir, '..', 'DataEngineering', 'ktm_all.db'))
print(f'Target database: {db_file}')

# ── Curated list of dining/cafe/restaurant categories ────────────────────────
TARGET_CATEGORIES = [
    'adventure sports center', 'american restaurant', 'animal cafe', 'asian restaurant',
    'assamese restaurant', 'bakery', 'banquet hall', 'bar', 'barbecue restaurant',
    'bed and breakfast', 'biryani restaurant', 'breakfast restaurant', 'brunch restaurant',
    'bubble tea store', 'cafe', 'candy store', 'chicken restaurant', 'chicken wings restaurant',
    "children's cafe", 'chinese noodle restaurant', 'chinese restaurant', 'coffee roasters',
    'coffee shop', 'coffee stand', 'coffee store', 'cosplay cafe', 'cottage rental',
    'dairy store', 'dance restaurant', 'dessert restaurant', 'dessert shop', 'dim sum restaurant',
    'dog cafe', 'door manufacturer', 'dumpling restaurant', 'english restaurant', 'espresso bar',
    'family restaurant', 'fast food restaurant', 'frozen food store', 'hamburger restaurant',
    'hong kong style fast food restaurant', 'ice cream shop', 'indian restaurant', 'internet cafe',
    'karaoke bar', 'korean restaurant', 'lounge', 'lunch restaurant', 'mandarin restaurant',
    'meat dish restaurant', 'momo restaurant', 'musical club', 'nepalese restaurant',
    'pizza restaurant', 'pub', 'restaurant', 'restaurant or cafe', 'rice restaurant',
    'southeast asian restaurant', 'southern restaurant (us)', 'spanish restaurant', 'sports bar',
    'takeout restaurant', 'tea house', 'tea store', 'thai restaurant', 'vegetarian restaurant',
    'wholesale bakery', 'wine bar',
]

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# ── Step 1: Find place_ids to delete ─────────────────────────────────────────
placeholders = ','.join(['?' for _ in TARGET_CATEGORIES])
cursor.execute(f"""
    SELECT DISTINCT p.place_id
    FROM place p
    JOIN categories_detailed cd ON p.place_id = cd.place_id
    WHERE (p.review_count IS NULL OR p.review_count = 0)
      AND LOWER(cd.category_name) IN ({placeholders})
""", TARGET_CATEGORIES)
ids_to_delete = [row[0] for row in cursor.fetchall()]
print(f'Places to remove: {len(ids_to_delete):,}')

if not ids_to_delete:
    print('Nothing to delete.')
else:
    # ── Step 2: Delete from all child tables, then place ─────────────────────
    CHILD_TABLES = [
        'categories_detailed',
        'images',
        'images_detailed',
        'open_hours_detailed',
        'owner_info',
        'popular_times_detailed',
        'ratings_distribution',
        'reviews',
        'reviews_detailed',
    ]

    # Batch deletes in chunks of 500 to avoid SQLite parameter limits
    CHUNK = 500
    for tbl in CHILD_TABLES:
        total_del = 0
        for i in range(0, len(ids_to_delete), CHUNK):
            chunk = ids_to_delete[i:i+CHUNK]
            ph = ','.join(['?' for _ in chunk])
            cursor.execute(f'DELETE FROM {tbl} WHERE place_id IN ({ph})', chunk)
            total_del += cursor.rowcount
        print(f'  Deleted {total_del:,} rows from {tbl}')

    # Delete from place table
    total_place_del = 0
    for i in range(0, len(ids_to_delete), CHUNK):
        chunk = ids_to_delete[i:i+CHUNK]
        ph = ','.join(['?' for _ in chunk])
        cursor.execute(f'DELETE FROM place WHERE place_id IN ({ph})', chunk)
        total_place_del += cursor.rowcount
    print(f'  Deleted {total_place_del:,} rows from place')

    conn.commit()

    # ── Step 3: Verify ───────────────────────────────────────────────────────
    print('\nVerification — remaining record counts:')
    for tbl in ['place'] + CHILD_TABLES:
        cursor.execute(f'SELECT COUNT(*) FROM {tbl}')
        print(f'  {tbl}: {cursor.fetchone()[0]:,}')

conn.close()
print('\nDone.')
